# Step 0 - Install and Import package

In [ ]:
!pip install Taulu

### Google Colab
If you are using Google Colab, first do the following:

1) Restart your runtime/session after installing Taulu, if you haven't already

2) Run the code block below

3) Proceed with importing taulu

In [ ]:
from google.colab import output
output.enable_custom_widget_manager()

### Importing Taulu

In [2]:
%matplotlib widget
from taulu import Taulu

## Step 1 — Annotate the header

Taulu needs to know where the column boundaries are. Use `Taulu.annotate()` to open an interactive widget where you draw a box around **one header row** of the table.

For tables that span two pages (left + right), annotate each page separately. The two annotations are later combined using `Split`.

# Taulu Demo

**Taulu** segments historical tabular images into individual cells by detecting the grid structure of ruled tables.

This demo walks through the full pipeline:
1. **Annotate** the table header to teach Taulu the column layout
2. **Segment** the table — Taulu detects and grows the grid automatically
3. **Inspect** the detected cells interactively
4. **Save** the result for downstream use (e.g. OCR)

> **Prerequisite:** Run `%matplotlib widget` (already included below) to enable the interactive annotation and cell viewer widgets inside Jupyter.

In [ ]:
annotation_left = Taulu.annotate("../data/table_00.png", "table_00_left.png")

In [ ]:
annotation_right = Taulu.annotate("../data/table_00.png", "table_00_right.png")

When you are done annotating, click **Done** in the widget, then run the cell below to save both annotations to disk.

In [ ]:
# Run this cell AFTER clicking "Done" on the annotation widget
annotation_left.save("table_00_left.json")
annotation_right.save("table_00_right.json")

## Step 2 — Configure and segment the table

Pass the saved annotation files to `Taulu` along with tuning parameters, then call `segment_table()` on the source image. The parameters below are explained in the reference section that follows; the defaults work well for most clean scans.

### Taulu Configuration Parameters
#### Image & Template Paths
**header_image_path**: Path to your annotated header template image. For two-page tables (where the table spans across left and right pages), wrap paths in `Split("left.png", "right.png")`.

**header_anno_path**: Explicit path to the JSON annotation file. If not provided, Taulu infers it from the image path (e.g., `header.png` → `header.json`).

#### Row Height Configuration

**cell_height_factor**: Multiplier to calculate data row heights from the header row height. For example, `0.8` means data rows are 80% the height of the header row. Pass a list like `[1.0, 0.8]` if different row types have different heights. Default: `[1.0]`

#### Binarization (Image Preprocessing)
**sauvola_k**: Threshold parameter for Sauvola adaptive binarization (0.0–1.0). Higher values make thresholding more aggressive, removing more noise but potentially losing faint lines. Increase if you see noise interfering with line detection. Default: `0.25`

#### Corner Detection
**search_region**: Size in pixels of the square area searched when looking for the next grid corner. Increase for documents with more warping/distortion, decrease for cleaner scans with straight lines. Default: `60`

**distance_penalty**: Weight (0–1) that penalizes corner candidates further from the expected position. Higher values favor corners closer to predicted locations (more conservative). Lower values allow more deviation (useful for warped documents). Default: `0.4`

**grow_threshold**: Confidence threshold (0–1) for accepting a detected corner during grid expansion. Lower values accept weaker matches, higher values require stronger corner evidence. Default: `0.3`

#### Cross-Kernel (Corner Template Matching)
**kernel_size**: Size of the cross-shaped kernel used to detect rule intersections. Must be odd. Larger kernels are more selective but may miss corners in low-contrast areas. Default: `41`

**cross_width**: Width in pixels of each arm in the cross kernel. Should approximately match the thickness of table rules (lines) *after* morphological dilation. Default: `10`

**morph_size**: Size of the morphological dilation structuring element. Dilation "thickens" and connects broken line segments. Increase for documents with fragmented lines, but note this also thickens rules. Default: `4`

#### Processing & Performance

**processing_scale**: Downscale factor (0–1] for image processing. Values less than 1.0 speed up processing significantly but may reduce accuracy. Use `0.5` to process at half resolution. Default: `1.0`

#### A* Pathfinding
**skip_astar_threshold**: Confidence threshold (0–1) above which A* pathfinding is skipped in favor of the faster heuristic jump. Higher values skip A* more often (faster but less accurate for difficult cases). Default: `0.2`

#### Table Growing Algorithm
**min_rows**: Minimum number of data rows to detect before the algorithm is allowed to stop. Prevents premature termination on tables with many rows. Default: `5`

**look_distance**: Number of previously detected rows to examine when extrapolating the next row position. Higher values produce smoother extrapolation but react slower to changes in row spacing. Default: `3`

#### Grid Refinement
**smooth_grid**: If `True`, applies smoothing to the detected grid after detection to reduce jagged edges. Default: `False`

**cuts**: Number of "cut" operations during grid growing. Cuts remove poorly-placed corners and re-grow them, improving overall grid quality. Default: `3`

**cut_fraction**: Fraction (0–1) of detected corners to remove during each cut operation. Higher values are more aggressive, removing more points for re-detection. Default: `0.5`



### Debug views

When `debug_view_notebook=True`, `segment_table()` displays four intermediate images in order:

| View | What you see | What to check |
|------|-------------|---------------|
| **thresholded** | Binary (black/white) version of the table image produced by Sauvola adaptive binarization | Table rules should appear as clear white lines on a black background. If rules are broken or noisy, adjust `sauvola_k`. |
| **dilated** | Thresholded image after morphological dilation | Rule segments that were fragmented should now be connected. If rules are still broken, increase `morph_size`. |
| **filtered** | Result of cross-kernel template matching — bright spots mark detected rule intersections | Bright peaks should sit exactly at grid corners. Adjust `kernel_size` and `cross_width` to match the actual line thickness. |
| **matches** | ORB feature matches between the annotated header template and the target image | Lines connect matched keypoints. A healthy result shows roughly parallel lines of similar length. Scattered or crossing lines indicate a poor alignment — try adjusting `sauvola_k` on the `Taulu` constructor. |

In [ ]:
from taulu import Split

# Note that if you use split, the image will be processed twice

taulu = Taulu(
    header_image_path=Split("table_00_left.png", "table_00_right.png"),
    sauvola_k=0.25,
    look_distance=30,
    morph_size=7,
    kernel_size=35,
    search_region=30,
    cell_height_factor=0.15,
    min_rows=45
)
table = taulu.segment_table("../data/table_00.png", debug_view_notebook=True)

## Step 3 — Inspect the detected cells

`show_cells()` opens an interactive viewer. Click any cell in the image to highlight it and confirm the grid was detected correctly before proceeding.

In [ ]:
table.show_cells("../data/table_00.png")

## Step 4 — Save the result

`table.save()` writes the detected grid corners to a JSON file. This file can be reloaded later with `TableGrid.from_saved()` to crop individual cells without re-running the segmentation.

In [48]:
# Save
table.save("tables_02_cells.json")